# Explaining the 8-Bit Float Format

## Open Compute Project FP8 Specification

OFP8 representation consists of sign, exponent, and mantissa fields. In this specification we
use the term mantissa to refer to the trailing significand bits. Two encodings are defined - E4M3
and E5M2, where the name explicitly states the number of bits in the exponent (E) and mantissa
(M) fields. Encodings consist of:

- 1 sign bit: the most significant bit
- e-bit biased exponent: 4 bits for E4M3, 5 bits for E5M2
- m mantissa (trailing significand) bits: 3 bits for E4M3, 2 bits for E5M2

The value, v, of a normal OFP8 number is
$$ v = (−1)^S × 2^{E−bias} × (1 + 2^{−m} × M) $$
The value, v, of a subnormal OFP8 number (subnormals have E = 0 and M > 0) is
$$ v = (−1)^S × 2^{1−bias} × (0 + 2^{−m} × M) $$

Exponent parameters and min/max values for both OFP8 formats are specified in Table 1.
The E5M2 format represents infinities and NaNs. Interpretation of the three mantissa values for
NaNs is not defined. The E4M3 format does not represent infinities and uses only two bit
patterns for NaN (a single mantissa-exponent bit pattern but allowing both values of the sign bit)
in order to increase emax to 8 and thus to increase the dynamic range by one binade. Various
values for OFP8 formats are detailed in Table 2.

### Table 1: OFP8 exponent parameters

| Parameter       | E4M3 | E5M2 |
| --------------- | ---- | ---- |
| Exponent bias   | 7    | 15   |
| emax (unbiased) | 8    | 15   |
| emin (unbiased) | -6   | -14  |

### Table 2: OFP8 value encoding details

| Parameter            | E4M3                        | E5M2                        |
| -------------------- | --------------------------- | --------------------------- |
| Infinities           | N/A                         | S.11111.00₂                 |
| NaN                  | S.1111.111₂                 | S.11111.{01, 10, 11}₂       |
| Zeros                | S.0000.000₂                 | S.00000.00₂                 |
| Max normal number    | S.1111.110₂ = ±448          | S.11110.11₂ = ±57,344       |
| Min normal number    | S.0001.000₂ = ±2⁻⁶          | S.00001.00₂ = ±2⁻¹⁴         |
| Max subnormal number | S.0000.111₂ = ±0.875 \* 2⁻⁶ | S.00000.11₂ = ±0.75 \* 2⁻¹⁴ |
| Min subnormal number | S.0000.001₂ = ±2⁻⁹          | S.00000.01₂ = ±2⁻¹⁶         |
| Dynamic range        | 18 binades                  | 32 binades                  |


## Standard Floating point multiplication

A floating point number can be represented with a sign bit $s$, exponent bits $e$, bias $b$ and mantissa bits $m$ where the value $v$ is is calculated as:

$$
v = (-1)^{\textstyle s} \times 2^{\textstyle (e-b)} \times (1+m) \\
$$

We can use the following equation to multiply two floats together:

$$
p = (s_1 \oplus s_2) \times 2^{\textstyle (e_1+e_2-b)} \times (1+m_1+m_2+m_1 m_2)
$$

Since the exponent bit representations ($e_1$ and $e_2$) are biased, we need to subtract the bias ($b$) to get the actual exponent value, otherwise the bias is counted twice.

Since `float8` is not a common or built-in datatype, we wrote a custom class for working with the format, lets import that

In [1]:
from float8 import Float8